# 09 — Pilotdaten der Zielhardware

Entspricht Kapitel 5, Abschnitt *Validierung auf der Zielhardware*.

Erstmals werden Daten ausgewertet, die mit der **tatsächlichen
Zielhardware** aufgezeichnet wurden (ESP32-C6 + MPU-6500). Alle
bisherigen Ergebnisse beruhen auf öffentlichen Datensätzen mit
fremder Sensorik.

**Aufgezeichnet an zwei Positionen:**

| Position | Beantwortet |
|---|---|
| Taille | Position wie SisFall — verbleibender Unterschied ist allein die Hardware |
| Fuß | Zielposition — Hardware und Position gemeinsam |

Ohne die Taillenaufnahme ließe sich ein schlechtes Fußergebnis nicht
zuordnen: Es bliebe offen, ob die Sensorik oder die Anbringung die
Ursache ist.

**Einschränkung:** Die Aufnahmen stammen von einer einzigen Person.
Eine subjektunabhängige Bewertung wie bei den öffentlichen
Datensätzen ist damit nicht möglich. Die Zero-Shot-Auswertung bleibt
gültig, da das Modell diese Daten nie gesehen hat; für das
Fine-Tuning wird nach Aufnahmen getrennt, was schwächer ist als eine
Trennung nach Probanden.

## Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q scipy scikit-learn tensorflow

import os, sys, shutil, time, json, re
from pathlib import Path
import numpy as np
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

!wget -q --no-cache https://raw.githubusercontent.com/Alyelb/ML-Modell-for-Fall-detection/main/fall_lib.py -O /content/fall_lib.py
sys.path.insert(0, "/content")
import fall_lib as fl

BASE  = "/content/drive/MyDrive/ML_Fall_Detection"
fl.RESULTS_ROOT = f"{BASE}/results"
MODEL = f"{BASE}/results/fall_model.keras"
MODEL_LL = f"{BASE}/results/fall_model_lowerlimb.keras"   # aus Notebook 10
PILOT = f"{BASE}/Pilotdaten"          # <-- CSV-Dateien hier ablegen

run = fl.new_run("09_pilotdaten")
print("Konfiguration:", fl.default_config())

## Aufnahmen einlesen

Der Logger speichert ADC-Rohwerte mit 100 Hz. Umrechnung und
Heruntertasten auf 50 Hz erfolgen hier mit **denselben Funktionen wie
beim Training** -- nur so ist die Vorverarbeitung identisch.

Sturzdateien dürfen mehrere Wiederholungen enthalten (z. B. fünf
Stürze nach vorn in einer Aufnahme). Jeder Aufprall wird als eigener
Sturz-Trial ausgewertet; die Anzahl je Datei wird unten ausgegeben und
sollte mit dem Protokoll übereinstimmen.

In [ ]:
# Empfindlichkeiten laut Konfiguration des Loggers
ACC_LSB = 1.0 / 2048.0      # +-16 g
GYR_LSB = 1.0 / 16.384      # +-2000 deg/s
FS_ROH  = 100

STURZ_MARKER = ("fall_",)   # Aktivitaeten, die als Sturz gelten


def lade_pilot(ordner):
    """Liest die CSV-Dateien des Loggers.

    Kopfzeile: # position=WAIST activity=walking fs=100 ...
    Datenzeilen: t_ms,ax,ay,az,gx,gy,gz  (LSB)
    """
    trials = []
    dateien = sorted(Path(ordner).rglob("*.csv"))
    if not dateien:
        raise FileNotFoundError(f"Keine CSV in {ordner}")

    for f in dateien:
        text = f.read_text(errors="ignore").splitlines()
        meta = {}
        for zeile in text[:5]:
            if zeile.startswith("#"):
                for teil in zeile.lstrip("#").split():
                    if "=" in teil:
                        k, v = teil.split("=", 1)
                        meta[k] = v

        # Aus dem Dateinamen ergaenzen, falls die Kopfzeile fehlt
        position = meta.get("position")
        aktivitaet = meta.get("activity")
        if position is None or aktivitaet is None:
            teile = f.stem.split("_", 2)
            if len(teile) >= 3:
                position   = position or teile[1]
                aktivitaet = aktivitaet or teile[2]

        werte = []
        for zeile in text:
            zeile = zeile.strip()
            if not zeile or zeile.startswith("#") or zeile.startswith("t_ms"):
                continue
            teile = zeile.split(",")
            if len(teile) < 7:
                continue
            try:
                werte.append([float(x) for x in teile[:7]])
            except ValueError:
                continue

        if len(werte) < FS_ROH:
            print(f"  uebersprungen (zu kurz): {f.name}")
            continue

        arr = np.asarray(werte, dtype=np.float64)
        t   = arr[:, 0]
        daten = np.hstack([arr[:, 1:4] * ACC_LSB,
                           arr[:, 4:7] * GYR_LSB]).astype(np.float32)

        # Tatsaechliche Rate aus den Zeitstempeln bestimmen
        dauer = (t[-1] - t[0]) / 1000.0
        fs_ist = len(t) / dauer if dauer > 0 else FS_ROH

        # Schreibzugriffe auf den Flash verzoegern einzelne Abtastungen
        # (Luecken bis ~80 ms, danach holt der Logger auf). Die
        # Gesamtzahl stimmt, die Zeitachse ist aber ungleichmaessig.
        # Daher auf ein exaktes 100-Hz-Raster interpolieren, bevor
        # heruntergetastet wird.
        t, idx = np.unique(t, return_index=True)
        daten = daten[idx]
        raster = np.arange(t[0], t[-1], 1000.0 / FS_ROH)
        daten = np.column_stack([np.interp(raster, t, daten[:, k])
                                 for k in range(6)]).astype(np.float32)

        daten = fl.downsample(daten, FS_ROH, fl.FS_TARGET)
        label = 1 if any(m in (aktivitaet or "") for m in STURZ_MARKER) else 0

        basis = {"subject": position,          # dient hier der Gruppierung
                 "activity": aktivitaet or "unbekannt",
                 "position": position or "unbekannt",
                 "datei": f.name, "fs_ist": fs_ist}

        if label == 0:
            trials.append({**basis, "data": daten, "label": 0,
                           "impact_idx": None})
            continue

        # Sturzdatei: mehrere Wiederholungen je Datei moeglich.
        # Jeder Aufprall wird ein eigener Trial (+-5 s), damit nicht
        # nur der staerkste Sturz der Datei ausgewertet wird.
        for k, (s, e, imp) in enumerate(aufpralle(daten)):
            trials.append({**basis, "data": daten[s:e], "label": 1,
                           "impact_idx": imp, "wiederholung": k + 1})
    return trials


def aufpralle(daten, kontext_s=5.0, min_abstand_s=6.0, min_g=2.0,
              rel=0.4):
    """Findet die Aufpralle einer Sturzdatei.

    Spitzen von |a| ueber max(min_g, rel * Dateimaximum) im Abstand von
    mindestens min_abstand_s. Liefert (start, ende, impact_idx relativ).
    Wird nichts gefunden, gilt das globale Maximum als einziger
    Aufprall.
    """
    from scipy.signal import find_peaks
    fs = fl.FS_TARGET
    mag = fl.acc_magnitude(daten)
    hoehe = max(min_g, rel * float(mag.max()))
    peaks, _ = find_peaks(mag, height=hoehe, distance=int(min_abstand_s * fs))
    if len(peaks) == 0:
        peaks = [int(np.argmax(mag))]
    ctx, out = int(kontext_s * fs), []
    for p in peaks:
        s, e = max(0, p - ctx), min(len(daten), p + ctx)
        if e - s >= fl.WIN_LEN:
            out.append((s, e, int(p - s)))
    return out


alle = lade_pilot(PILOT)

from collections import Counter
print(f"\n{len(alle)} Aufnahmen gelesen")
print(f"  Positionen : {dict(Counter(t['position'] for t in alle))}")
print(f"  Klassen    : {dict(Counter('Sturz' if t['label'] else 'ADL' for t in alle))}"
      "  (Sturz = erkannte Aufpralle)")
for f_ in sorted({t['datei'] for t in alle if t['label']}):
    n_ = sum(1 for t in alle if t['datei'] == f_)
    print(f"    {f_:<36} {n_} Aufprall(e)")
raten = [t['fs_ist'] for t in alle]
print(f"  Abtastrate : {np.mean(raten):.1f} Hz "
      f"(min {np.min(raten):.1f}, max {np.max(raten):.1f})")
if np.mean(raten) < 95 or np.mean(raten) > 105:
    print("  ACHTUNG: Rate weicht deutlich von 100 Hz ab -- "
          "Auswertung mit Vorsicht betrachten.")

## Signalvergleich mit SisFall

Vor jeder Modellauswertung ein Blick auf die Rohsignale: Weichen die
Spitzenwerte stark ab, ist ein schlechtes Transferergebnis schon hier
erkennbar und muss nicht dem Modell angelastet werden.

In [ ]:
SIS = f"{BASE}/SisFall_dataset"
sis_lokal = "/content/SisFall"
if not Path(sis_lokal).exists() and Path(SIS).exists():
    shutil.copytree(SIS, sis_lokal)

sis = fl.load_sisfall(sis_lokal, verbose=False) if Path(sis_lokal).exists() else []

def spitzen(trials, filt=None):
    p_f, p_a = [], []
    for t in trials:
        if filt and not filt(t):
            continue
        m = float(fl.acc_magnitude(t["data"]).max())
        (p_f if t["label"] == 1 else p_a).append(m)
    return np.array(p_f), np.array(p_a)

print(f"{'Quelle':<26} {'Stuerze':>22} {'ADL':>22}")
print(f"{'':26} {'Median   p95':>22} {'Median   p95':>22}")
print("-"*72)

zeilen = [("SisFall (Huefte)", sis, None)]
for pos in sorted({t["position"] for t in alle}):
    zeilen.append((f"Pilot ({pos})", alle,
                   lambda t, p=pos: t["position"] == p))

signal_stats = {}
for name, trials, filt in zeilen:
    if not trials:
        continue
    pf, pa = spitzen(trials, filt)
    mf = f"{np.median(pf):6.2f} {np.percentile(pf,95):7.2f}" if len(pf) else "     --      --"
    ma = f"{np.median(pa):6.2f} {np.percentile(pa,95):7.2f}" if len(pa) else "     --      --"
    print(f"{name:<26} {mf:>22} {ma:>22}")
    signal_stats[name] = {
        "n_falls": int(len(pf)), "n_adl": int(len(pa)),
        "fall_median_g": round(float(np.median(pf)), 3) if len(pf) else None,
        "adl_p95_g": round(float(np.percentile(pa, 95)), 3) if len(pa) else None,
    }

## Zero-Shot-Auswertung

Beide Modelle werden unverändert auf die Pilotdaten angewendet:

| Modell | Training |
|---|---|
| **Basis** | SisFall (Hüfte) |
| **Lower-Limb** | Basis → KFall → CAPPIMU-Knie + ImbalanSense-Fuß (Notebook 10) |

Die Auswertung ist uneingeschränkt gültig: Keines der Modelle hat
diese Aufnahmen gesehen. Der Vergleich zeigt, ob beinnahe
Trainingsdaten den Transfer auf den Fuß verbessern.

In [ ]:
from tensorflow import keras

def lade_modell(pfad):
    m = keras.models.load_model(pfad, compile=False)
    m.compile(optimizer="adam", loss="binary_crossentropy",
              metrics=["accuracy"])
    return m

MODELLE = {"Basis": lade_modell(MODEL)}
if Path(MODEL_LL).exists():
    MODELLE["Lower-Limb"] = lade_modell(MODEL_LL)
else:
    print("Hinweis: Lower-Limb-Modell fehlt -- erst Notebook 10 ausfuehren.")
model = MODELLE["Basis"]          # Fine-Tuning unten startet vom Basismodell
print("Modelle:", list(MODELLE), "\n")

zero_shot = {}
for pos in sorted({t["position"] for t in alle}):
    teil = [t for t in alle if t["position"] == pos]
    X, y, g, akt = fl.build_windows(teil, mode="standard")
    if len(y) == 0 or y.sum() == 0:
        print(f"{pos}: keine Sturzfenster -- uebersprungen")
        continue
    Xn = fl.znorm(X)
    print(f"{pos}  ({len(y)} Fenster, {int(y.sum())} Stuerze)")
    for name, m in MODELLE.items():
        r = fl.evaluate(m, Xn, y, f"  {name}")
        r["Se_ci95"] = fl.wilson_ci(r["TP"], r["TP"] + r["FN"])
        zero_shot[f"{pos}|{name}"] = r


### Falsch-Positiv-Rate auf den Alltagsaufnahmen

Für den Betrieb ist die Zahl der Fehlalarme je Stunde die
entscheidende Größe.

In [ ]:
SEK_PRO_FENSTER = fl.WIN_SEC * (1 - fl.OVERLAP)
fp_stats = {}

for pos in sorted({t["position"] for t in alle}):
    adl = [t for t in alle if t["position"] == pos and t["label"] == 0]
    if not adl:
        continue
    X, y, g, akt = fl.build_windows(adl, mode="sliding")
    Xn = fl.znorm(X)
    std = len(X) * SEK_PRO_FENSTER / 3600
    print(f"\n{pos}: {len(X)} ADL-Fenster = {std:.2f} Stunden")

    for name, m in MODELLE.items():
        p = m.predict(Xn, batch_size=256, verbose=0).flatten()
        print(f"\n  [{name}]  {'Schwelle':>9} {'FP':>6} {'Rate':>8} {'FP/Std':>9}")
        eintrag = {}
        for thr in [0.5, 0.7, 0.9, 0.95]:
            fp = int((p > thr).sum())
            print(f"  {'':{len(name)+2}}  {thr:>9.2f} {fp:>6d} {100*fp/len(p):>7.2f}% "
                  f"{fp/std if std else 0:>9.1f}")
            eintrag[str(thr)] = {"fp": fp, "rate": round(fp/len(p), 4),
                                 "fp_per_hour": round(fp/std, 2) if std else None}
        fp_stats[f"{pos}|{name}"] = eintrag

        print(f"    nach Aktivitaet (Schwelle 0.9):")
        for a in sorted(np.unique(akt)):
            pa = p[akt == a]
            print(f"      {a:<18} {len(pa):>5} Fenster  {100*(pa>0.9).mean():>6.2f}%")


## Fine-Tuning auf den Pilotdaten

Getrennt wird nach Aufnahmen, nicht nach Probanden -- bei einer
einzigen Person ist mehr nicht möglich. Das Ergebnis ist daher
optimistischer zu lesen als die Werte der öffentlichen Datensätze.

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

fine_tuned = {}
for pos in sorted({t["position"] for t in alle}):
    teil = [t for t in alle if t["position"] == pos]
    if sum(t["label"] for t in teil) < 6:
        print(f"{pos}: zu wenige Stuerze fuer Fine-Tuning")
        continue

    X, y, _, _ = fl.build_windows(teil, mode="standard")
    Xn = fl.znorm(X)
    # Gruppierung nach Aufnahme, damit Fenster einer Aufnahme nicht
    # gleichzeitig in Training und Test landen
    gruppen = []
    for t in teil:
        n = len(fl.window_trial(t["data"], t["label"]))
        gruppen += [t["datei"]] * n
    gruppen = np.asarray(gruppen)

    gss = GroupShuffleSplit(n_splits=1, test_size=0.35, random_state=42)
    tr, te = next(gss.split(Xn, y, gruppen))
    if y[te].sum() == 0:
        print(f"{pos}: Testsatz ohne Stuerze -- uebersprungen")
        continue

    print(f"\n{pos}: Train {len(tr)} | Test {len(te)} "
          f"({int(y[te].sum())} Stuerze)")
    m = fl.finetune(model, Xn[tr], y[tr])
    fine_tuned[pos] = fl.evaluate(m, Xn[te], y[te], "  Fine-Tuned")
    m.save(run / f"modell_{pos}.keras")

## Zusammenfassung

In [ ]:
print("="*74)
print("  PILOTDATEN -- ZIELHARDWARE ESP32-C6 + MPU-6500")
print("="*74)
print(f"\n{'Position':<14} {'Modell':<12} {'Zero-Shot Se':>13} {'95%-KI':>17} {'FP/Std @0.9':>13}")
print("-"*72)
for key in sorted(zero_shot):
    pos, name = key.split("|")
    r = zero_shot[key]
    fp = fp_stats.get(key, {}).get("0.9", {}).get("fp_per_hour")
    ci = r["Se_ci95"]
    print(f"{pos:<14} {name:<12} {r['Se']:>13.3f}   [{ci[0]:.2f}; {ci[1]:.2f}] "
          f"{(f'{fp:.1f}' if fp is not None else '--'):>13}")

if fine_tuned:
    print("\nFine-Tuning auf Pilotdaten (Basismodell, Trennung nach Aufnahme):")
    for pos, r in fine_tuned.items():
        print(f"  {pos:<14} Se={r['Se']:.3f}")

print("\nZum Vergleich (oeffentliche Datensaetze, Zero-Shot Basismodell):")
print("  KFall (unt. Ruecken, MPU-fremd) 1,000")
print("  FallAllD (Taille, LSM9DS1)      0,136")
print("  UMAFall (Knoechel, 20 Hz)       0,194")

ergebnisse = {
    "n_aufnahmen": len(alle),
    "modelle": list(MODELLE),
    "signalvergleich": signal_stats,
    "zero_shot": zero_shot,
    "fp_rate": fp_stats,
    "fine_tuned": fine_tuned,
    "einschraenkung": ("Aufnahmen einer einzigen Person; Trennung "
                       "nach Aufnahme statt nach Proband. Zero-Shot "
                       "uneingeschraenkt gueltig, Fine-Tuning "
                       "optimistisch."),
}
fl.save_results(run, ergebnisse, config=fl.default_config(),
    notes="Auswertung auf der Zielhardware, Basis- und Lower-Limb-Modell.")
